In [2]:
%load_ext autoreload
%autoreload 2
import mattertune as mt
from mattertune import MatterTuner
import mattertune.configs as MC
import os
from pathlib import Path
import ase
from mattertune.backbones import (
    EqV2BackboneModule,
    JMPBackboneModule,
    ORBBackboneModule,
)
from mattertune.configs import WandbLoggerConfig

/global/cfs/projectdirs/m3641/Aamod/MatterTune_diffusion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/global/cfs/projectdirs/m3641/Aamod/MatterTune_diffusion/.venv/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [30]:
def hparams(args_dict):
    hparams = MC.MatterTunerConfig.draft()

    hparams.model = MC.ORBBackboneConfig.draft()
    hparams.model.pretrained_model = args_dict["model_type"]

    hparams.model.ignore_gpu_batch_transform_error = True
    hparams.model.freeze_backbone = False
    hparams.model.reset_output_heads = True
    hparams.model.optimizer = MC.AdamWConfig(
        lr=5e-5,
        weight_decay=0,
    )

    # Add model properties
    hparams.model.properties = []
    property = MC.NoisePropertyConfig(
        loss = MC.MSELossConfig(),
        dtype= "float",
        name="noise",
    )

    hparams.model.properties.append(property)

    ## Data Hyperparameters
    hparams.data = MC.ManualSplitDataModuleConfig.draft()
    hparams.data.train = MC.XYZDatasetConfig.draft()

    hparams.data.train.T = args_dict["T"]
    hparams.data.train.sigma_min = args_dict["sigma_min"]
    hparams.data.train.sigma_max= args_dict["sigma_max"]
    hparams.data.train.diffusion_type = "subvp"

    hparams.data.train.src = "./data/relaxed.xyz"
    hparams.data.batch_size = args_dict["batch_size"]
    hparams.data.num_workers = 0
    hparams.data.pin_memory = False

    ## Trainer Hyperparameters
    hparams.trainer = MC.TrainerConfig.draft()
    hparams.trainer.max_epochs = args_dict["max_epochs"]
    hparams.trainer.accelerator = "gpu"
    hparams.trainer.devices = args_dict["devices"]
    hparams.trainer.gradient_clip_algorithm = "norm"
    hparams.trainer.gradient_clip_val = 1.0
    hparams.trainer.precision = "32"

    # Configure Model Checkpoint
    ckpt_name = f"{args_dict['model_type']}-best"
    if os.path.exists(f"./checkpoints/{ckpt_name}.ckpt"):
        os.remove(f"./checkpoints/{ckpt_name}.ckpt")
    hparams.trainer.checkpoint = MC.ModelCheckpointConfig(
        dirpath="./checkpoints",
        filename=ckpt_name,
        save_top_k=1,
        mode="min",
        every_n_epochs=100,
    )

    #Configure Logger
    # hparams.trainer.loggers = [
    #     WandbLoggerConfig(
    #         project="Diffusion-Pretraining-Orb_v2-new",
    #         name=f"Pretrain-diffusion-{args_dict['model_type']}",
    #     )
    # ]

    # Additional trainer settings
    hparams.trainer.additional_trainer_kwargs = {
        "inference_mode": False,
    }

    hparams = hparams.finalize(strict=False)
    return hparams

args_dict = {
    "model_type" : "orb-v2",
    "batch_size" : 32,
    "devices" : [0],
    "max_epochs" : 200,
    "T": 32,
    "sigma_min": 0.05,
    "sigma_max": 2
}

In [31]:
mt_config = hparams(args_dict)
model, trainer = MatterTuner(mt_config).tune()
trainer.save_checkpoint(f"./cpkts/diffusion_T-{args_dict["T"]}_sigmamin-{args_dict["sigma_min"]}_sigmamax-{args_dict["sigma_max"]}.cpkt")

Returning model without loading pretrained weights
Epoch 0:   0%|          | 0/1962 [00:00<?, ?it/s]

Pred mean/std: 0.2140121 0.60170007
Epoch 0:   0%|          | 1/1962 [00:00<07:50,  4.17it/s, v_num=32]

Pred mean/std: -1.9573631 2.2155907
Epoch 0:   0%|          | 2/1962 [00:00<05:24,  6.04it/s, v_num=32]

Pred mean/std: -0.339554 0.9451365
Epoch 0:   0%|          | 3/1962 [00:00<05:42,  5.71it/s, v_num=32]

Pred mean/std: 0.5725538 0.8592274
Epoch 0:   0%|          | 4/1962 [00:00<05:49,  5.60it/s, v_num=32]

Pred mean/std: 0.17769694 0.5480783
Epoch 0:   0%|          | 5/1962 [00:00<05:37,  5.79it/s, v_num=32]

Pred mean/std: -0.2079224 0.49287015
Epoch 0:   0%|          | 6/1962 [00:01<05:40,  5.74it/s, v_num=32]

Pred mean/std: -0.20664373 0.4517621
Epoch 0:   0%|          | 7/1962 [00:01<06:17,  5.18it/s, v_num=32]

Pred mean/std: -0.008439452 0.27708155
Epoch 0:   0%|          | 8/1962 [00:01<06:22,  5.11it/s, v_num=32]

Pred mean/std: 0.17528032 0.49273413
Epoch 0:   0%|    

SIGTERMException: 

In [ ]:
import random
from ase.io import read, write
from ase import Atoms

def mix_xyz_random(file1, file2, out_file="mixed.xyz", seed=None, frame1=-1, frame2=-1):
    if seed is not None:
        random.seed(seed)

    a1 = read(file1, index=frame1)  # ASE Atoms
    a2 = read(file2, index=frame2)

    combined = a1 + a2  # concatenates Atoms (symbols + positions, etc.)

    idx = list(range(len(combined)))
    random.shuffle(idx)

    mixed = combined[idx]  # ASE supports fancy indexing -> reordered Atoms

    # Keep a useful comment line
    mixed.info["comment"] = f"Random mix of {file1} + {file2} (seed={seed})"

    write(out_file, mixed)  # ASE writes valid XYZ
    return mixed

# Example
mix_xyz_random("./data/relaxed.xyz", "./data/synth.xyz", out_file="mixed.xyz", seed=42)


['Li     -0.754246935604086      5.277433431640660     -0.022268351785551\n',
 'Li      0.705964849550840      8.182230447895437     -0.522484135543745\n',
 'Y      -8.974732593520718      5.982322654847807      7.570742469744362\n',
 'Y      -6.517787876543492      4.466597078220468      4.243084463292842\n',
 'Y      -5.947056899909762      8.051484792638027      3.059776954815820\n',
 'Y      -4.376645821237590     13.188894789341019      7.545092775080804\n',
 'Y      -4.749511712416182      9.732208342287519      7.380385055506909\n',
 'Y      -4.484549483540659      6.292899908919356      7.501760229303574\n',
 'Y      -4.301360990428477      3.084089052557986      7.979498652665377\n',
 'Y      -2.502105020362288      4.825428086458423      3.540103911399613\n',
 'Y      -1.948921766198321      8.501942652603374      4.161919224262447\n',
 'Y      -2.256163408343661     11.890274589756631      3.509636158769540\n',
 'Y      -2.335440111087217      1.466889402190686      4.153070